# MiniMax H3 — A100 Optimization Benchmark

This notebook is separate from the main long-video notebook. Its purpose is to benchmark **speed vs quality** on an **A100 40 GB** using the same H3 Extender pipeline.

We compare:
- **0.90 MP**
- **0.60 MP**
- **0.40 MP**

All three keep:
- Ref2VA pruned INT8 ConvRot diffusion
- official Qwen3-VL **INT8 ConvRot** text encoder
- 4-step Turbo LoRA
- Euler + Simple
- same prompt, seed, refs, and clip duration

Recommended: **A100 + High RAM**.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PERSIST_MODELS_TO_DRIVE = False
PERSIST_OUTPUT_TO_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/MiniMax_H3_Optimization"

print("Drive root:", DRIVE_ROOT)


## 1. Clone the H3 branch


In [ ]:
%cd /content
!rm -rf /content/All-testing /content/minimax_h3_comfy
!git clone --depth 1 --branch minimax-h3-colab https://github.com/Logan17de/All-testing.git /content/All-testing
!cp -r /content/All-testing/video/minimax_h3_comfy /content/minimax_h3_comfy
%cd /content/minimax_h3_comfy
!ls -la


## 2. Install/update ComfyUI + H3 Extender


In [ ]:
import os
os.environ["COMFY_ROOT"] = "/content/ComfyUI"
os.environ["H3_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["H3_PERSIST_MODELS"] = "1" if PERSIST_MODELS_TO_DRIVE else "0"
os.environ["H3_PERSIST_OUTPUT"] = "1" if PERSIST_OUTPUT_TO_DRIVE else "0"
!bash install_comfy_h3.sh


## 3. Download the A100-oriented H3 model profile

This uses the pruned INT8 ConvRot Ref2VA model plus the official **INT8 ConvRot** Qwen3-VL encoder.


In [ ]:
MODEL_ROOT = f"{DRIVE_ROOT}/models" if PERSIST_MODELS_TO_DRIVE else "/content/ComfyUI/models"
!python download_models.py --profile ref2va-a100-int8 --model-root "$MODEL_ROOT"


## 4. Create three benchmark workflows


In [ ]:
import json, copy
from pathlib import Path

COMFY_ROOT = Path("/content/ComfyUI")
src = COMFY_ROOT / "custom_nodes/ComfyUI_MiniMax_H3_Extender/Workflow/MiniMax_Extender.json"
out_dir = COMFY_ROOT / "user/default/workflows"
out_dir.mkdir(parents=True, exist_ok=True)
base = json.loads(src.read_text(encoding="utf-8"))

CLIP = "qwen3vl_32b_minimax_h3_int8_convrot.safetensors"
UNET = "minimax_h3_ref2va_pruned_int8_convrot.safetensors"
LORA = "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"

def make_workflow(mp):
    data = copy.deepcopy(base)
    for node in data.get("nodes", []):
        t = node.get("type")
        vals = node.get("widgets_values")
        if not isinstance(vals, list):
            continue
        if t == "CLIPLoader" and vals:
            vals[0] = CLIP
        elif t == "UNETLoader" and vals:
            vals[0] = UNET
        elif t == "LoraLoaderModelOnly" and vals:
            vals[0] = LORA
            if len(vals) > 1: vals[1] = 1
        elif t == "ImageScaleToTotalPixels" and len(vals) >= 3:
            vals[0] = "nearest-exact"
            vals[1] = float(mp)
            vals[2] = 32
        elif t == "MiniMaxH3Extender" and len(vals) >= 10:
            vals[4] = 4
            vals[5] = "euler"
            vals[6] = "simple"
            vals[7] = 1
    name = f"MiniMax_H3_A100_{mp:.2f}MP.json"
    path = out_dir / name
    path.write_text(json.dumps(data, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")
    return path

for p in [make_workflow(mp) for mp in (0.90, 0.60, 0.40)]:
    print("Created:", p)


## 5. Launch ComfyUI


In [ ]:
%cd /content/minimax_h3_comfy
!bash launch_comfy.sh


## 6. Download the three workflows

Open each JSON in ComfyUI one at a time and use the same reference(s), prompt, seed, and duration.


In [ ]:
from google.colab import files
for name in [
    "MiniMax_H3_A100_0.90MP.json",
    "MiniMax_H3_A100_0.60MP.json",
    "MiniMax_H3_A100_0.40MP.json",
]:
    files.download(f"/content/ComfyUI/user/default/workflows/{name}")


## 7. Start resource monitoring before each generation

Run this immediately before clicking **Run** in ComfyUI.


In [ ]:
import subprocess, pathlib
MONITOR_DIR = pathlib.Path('/content/h3_bench')
MONITOR_DIR.mkdir(exist_ok=True)
monitor_py = MONITOR_DIR / 'monitor.py'
monitor_code = '''
import csv, time, subprocess, psutil, pathlib
p = pathlib.Path('/content/h3_bench/resource_log.csv')
with p.open('w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['timestamp','vram_used_mb','vram_total_mb','gpu_util_pct','ram_used_gb','ram_total_gb'])
    while True:
        try:
            out = subprocess.check_output(['nvidia-smi','--query-gpu=memory.used,memory.total,utilization.gpu','--format=csv,noheader,nounits'], text=True).strip().splitlines()[0]
            used,total,util = [float(x.strip()) for x in out.split(',')]
            vm = psutil.virtual_memory()
            w.writerow([time.time(), used, total, util, vm.used/1024**3, vm.total/1024**3])
            f.flush()
        except Exception:
            pass
        time.sleep(1)
'''
monitor_py.write_text(monitor_code)
subprocess.run(['pkill','-f',str(monitor_py)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
proc = subprocess.Popen(['python', str(monitor_py)])
(MONITOR_DIR / 'monitor.pid').write_text(str(proc.pid))
print('Monitoring started. PID:', proc.pid)
print('Now click Run in ComfyUI.')


## 8. Stop monitoring after generation finishes and summarize


In [ ]:
import os, signal, pathlib, pandas as pd
MONITOR_DIR = pathlib.Path("/content/h3_bench")
pid_file = MONITOR_DIR / "monitor.pid"
if pid_file.exists():
    try: os.kill(int(pid_file.read_text()), signal.SIGTERM)
    except Exception: pass
df = pd.read_csv(MONITOR_DIR / "resource_log.csv")
elapsed = df["timestamp"].iloc[-1] - df["timestamp"].iloc[0]
{
    "observed_seconds": round(float(elapsed), 1),
    "peak_vram_gb": round(float(df["vram_used_mb"].max()/1024), 2),
    "avg_gpu_util_pct": round(float(df["gpu_util_pct"].mean()), 1),
    "peak_gpu_util_pct": round(float(df["gpu_util_pct"].max()), 1),
    "peak_system_ram_gb": round(float(df["ram_used_gb"].max()), 2),
}


## 9. Record the result


In [ ]:
import pandas as pd
from pathlib import Path
RESULTS = Path("/content/h3_bench/results.csv")

# CHANGE THESE after each run:
megapixels = 0.90
clip_seconds = 10
generation_seconds = None
quality_score = None
notes = ""

row = pd.DataFrame([{
    "megapixels": megapixels,
    "clip_seconds": clip_seconds,
    "generation_seconds": generation_seconds,
    "sec_per_video_sec": (generation_seconds / clip_seconds) if generation_seconds else None,
    "quality_score_1_10": quality_score,
    "notes": notes,
}])
if RESULTS.exists():
    row = pd.concat([pd.read_csv(RESULTS), row], ignore_index=True)
row.to_csv(RESULTS, index=False)
row


## Fair benchmark rules

Do not change these between 0.90 / 0.60 / 0.40 MP:

- reference images
- prompt text
- seed
- duration
- 4 steps
- sampler/scheduler
- LoRA strength
- number of clips
- context settings

First benchmark **one 10-second clip**. After choosing the best MP setting, benchmark a **2 × 10-second continuous sequence**.
